# Train the PI-ARM for solving Diffusion equation

The 2D diffusion equation is:
$$\frac{\partial u}{\partial t}-\frac{\partial^{2}u}{\partial x^{2}}=f(t,x),$$
where $f(t,x)$ is the source term which was chosen equal to
$$f(t,x)=(\pi^{2}-1)\exp(-t)\sin(\pi x).$$
Additionally, $\Omega=[0,1]\times[0,1]$ is the chosen domain, with the following boundary conditions:
$$u(t = 0,x) = \sin(\pi x),$$
$$u(t,x = 0) = u(t,x = 1) = 0.$$
The exact solution of this PDE is
$$u(t,x) = \sin(\pi x)\exp(-t).$$

In [1]:
import torch
from rkan import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = RMultKAN(width=[2,5,1], grid=5, k=5, seed=2, device=device).speed()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

cuda
checkpoint directory created: ./model
saving model version 0.0


In [2]:
from torch import autograd
from tqdm import tqdm

# --- PDE and Solution Definitions ---
dim = 2
np_t = 100  # Number of sampling points in t dimension
np_x = 100  # # Number of sampling points in x dimension
np_b = 2000  # Number of sampling points per boundary
ranges_t = [0, 1]
ranges_x = [0, 1]

f_fun = lambda x: (torch.pi ** 2 - 1) * torch.exp(-x[:,[0]]) * torch.sin(torch.pi * x[:, [1]])
# exact solution
sol_fun = lambda x: torch.sin(torch.pi * x[:, [1]]) * torch.exp(-x[:,[0]])
init_fun = lambda x: torch.sin(torch.pi * x[:, [1]])
boundary_fun = lambda x: torch.zeros_like(x[:, [0]])

# Define the gradient calculation function
def batch_jacobian(func, x, create_graph=False):
    def _func_sum(x):
        return func(x).sum(dim=0)

    return autograd.functional.jacobian(_func_sum, x, create_graph=create_graph).permute(1, 0, 2)

# --- Data Sampling ---
# Internal sampling points
sampling_mode = 'random'
x_mesh = torch.linspace(ranges_t[0], ranges_t[1], steps=np_t)
y_mesh = torch.linspace(ranges_x[0], ranges_x[1], steps=np_x)
X, Y = torch.meshgrid(x_mesh, y_mesh, indexing="ij")
if sampling_mode == 'mesh':
    x_i = torch.stack([X.reshape(-1, ), Y.reshape(-1, )]).permute(1, 0)
else:
    x_i = torch.rand((np_t * np_x, 2))

# Sorted index
ids_t = torch.sort(x_i, dim=0)[1].transpose(0, 1)[0]
ids_x = torch.sort(x_i, dim=1)[1].transpose(0, 1)[0]
x_i = x_i.to(device)

# Boundary points
x_b = torch.rand((np_b, 2))

t_bound_left = torch.stack([torch.full_like(x_b[:, 0], 0), x_b[:, 1]], dim=1).to(device)  # t=0

x_bound_lower = torch.stack([x_b[:, 0], torch.full_like(x_b[:, 1], 0)], dim=1).to(device)  # x=0
x_bound_upper = torch.stack([x_b[:, 0], torch.full_like(x_b[:, 1], 1)], dim=1).to(device)   # x=1

# --- Training Parameters ---
steps = 1000
alpha = 0.01
log = 1
loss_history = []
num_interval = model.grid
interval_size = np_t * np_x // num_interval
sample_ratio = 0.3

best_l2 = float('inf')
init_epoch = 0

def train(start_grid_update_step=2000, stop_grid_update_step=4000, grid_update_num=10):
    pbar = tqdm(range(steps), desc='Training', ncols=100)
    
    for _ in pbar:
        def closure():
            global pde_loss, bc_loss, lap, loss
            optimizer.zero_grad()
    
            # Interior loss
            x_1 = x_i.clone().detach().requires_grad_(True)
            sol = model(x_1)
            sol_t = batch_jacobian(model, x_1, create_graph=True)[:, :, 0]
            sol_xx = batch_jacobian(lambda x: batch_jacobian(model, x, create_graph=True)[:, :, 1], x_1, create_graph=True)[:, :, 1]
            
            f_val = f_fun(x_1)
            lap = sol_t - sol_xx - f_val
            pde_loss = torch.mean(lap ** 2)
    
            # Boundary loss
            init_pred = model(t_bound_left)
            init_exact = init_fun(t_bound_left)
            
            bc_lower_pred = model(x_bound_lower)
            bc_upper_pred = model(x_bound_upper)
            
            bc_loss = torch.mean((init_pred - init_exact)**2) + \
                      torch.mean(bc_lower_pred ** 2) + torch.mean(bc_upper_pred ** 2)
                
            loss = alpha * pde_loss + bc_loss
            loss.backward()
            return loss
        
        # --- ADAPTIVE GRID UPDATE (AGU) MECHANISM ---
        grid_update_freq = int((stop_grid_update_step - start_grid_update_step) / grid_update_num)
        x_up = x_i.clone()
        # Check if the current iteration is within the adaptive update window
        if (_ + init_epoch) % grid_update_freq == 0 and \
            (_ + init_epoch) < stop_grid_update_step and \
                (_ + init_epoch) >= start_grid_update_step and \
                    len(loss_history) > 0:

            print(f"\n--- Iteration {(_ + init_epoch)}: Performing Adaptive Grid Update ---")
            try:
                # --- 1. Loss-Driven Resampling ---
                resample_points = []
                # Determine the total number of points to be adaptively resampled based on the sampling ratio
                total_resamples = int(np_t * np_x * sample_ratio)
                
                # Note: This implementation performs a simplified, dimension-wise partitioning.
                for idx in [ids_t, ids_x]:
                    x_subset = x_i[idx]
                    pde_residual_subset = lap[idx]
                    
                    # Partition the subset into intervals and compute the loss for each
                    sub_points = torch.chunk(x_subset, num_interval, dim=0)
                    sub_residual_sq = torch.chunk(pde_residual_subset**2, num_interval, dim=0)
                    sub_losses = torch.tensor([torch.sum(sub) for sub in sub_residual_sq], device=device)
                    total_subset_loss = torch.sum(sub_losses)
                    
                    # Calculate the sampling probability for each interval based on its relative loss
                    sub_probabilities = sub_losses / total_subset_loss
                    
                    # Determine the number of samples to draw from each interval
                    # We allocate half of the total resamples to each dimension's partitioning
                    interval_resamples = (sub_probabilities * (total_resamples / 2)).int()

                    # Perform the resampling for each interval
                    for i in range(num_interval):
                        num_to_sample = interval_resamples[i].item()
                        if num_to_sample == 0:
                            num_to_sample = 1 # Ensure at least one sample is drawn
                        
                        # Randomly draw samples from the interval
                        perm_indices = torch.randperm(sub_points[i].shape[0])[:num_to_sample]
                        resample_points.append(sub_points[i][perm_indices])
                
                # --- 2. Construct the Final Dataset for Grid Update ---
                # Combine all adaptively sampled points
                resample_points = torch.cat(resample_points, dim=0)
                
                # Construct a hybrid dataset: a mix of adaptively resampled points and randomly chosen points
                num_random_samples = (np_t * np_x) - resample_points.shape[0]
                random_indices = torch.randint(0, np_t * np_x, (num_random_samples,), device=device)
                x_up = torch.cat([resample_points, x_i[random_indices]], dim=0)

                # --- 3. Dynamic Mixing Coefficient Update ---
                # Adjust the grid blending based on the convergence trend
                loss_diff = loss_history[-1][0] - pde_loss.item()
                if abs(loss_diff) < 1e-3: # If loss is stable, favor the adaptive grid (exploitation)
                    model.grid_eps = max(0, model.grid_eps - 0.01)
                else: # If loss is changing, favor the uniform grid (exploration)
                    model.grid_eps = min(1, model.grid_eps + 0.01)
                
                # --- 4. Trigger the Grid Update in the Model ---
                model.update_grid_from_samples(x_up)

            except Exception as e:
                print(f"An error occurred during grid update: {e}. Skipping this step.")
                continue # Skip this update and proceed to the next iteration
       
        optimizer.step(closure)

        exact = sol_fun(x_i).to(device)
        pred_sol = model(x_i)
        l2 = torch.norm(exact - pred_sol) / torch.norm(exact)

        # auto save optimal model
        global best_l2
        if l2 < best_l2:
            best_l2 = l2
            model.saveckpt('./model/Dif_arkan')
        # Log progress
        if _ % log == 0:
            pbar.set_description("Lf: %.2e | Lb: %.2e | error: %.2e" % (
                pde_loss.cpu().detach().numpy(),
                bc_loss.cpu().detach().numpy(),
                l2.cpu().detach().numpy(),
            ))

        # Save loss history
        loss_history.append((pde_loss.cpu().detach().numpy(), bc_loss.cpu().detach().numpy(), loss.cpu().detach().numpy(), l2.cpu().detach().numpy()))

In [3]:
steps = 1000
for i in range(5):
    train(start_grid_update_step=500, stop_grid_update_step=3000, grid_update_num=20)
    init_epoch = init_epoch + steps
    r_l2 = [loss[3] for loss in loss_history]
    print(f"Epoch {np.argmin(r_l2)}: Minimum Rel-L2 error reduced to {np.min(r_l2)*100: .4f}%")
    for param_group in optimizer.param_groups:
        param_group['lr'] *= 0.7
        print(f"Epoch {init_epoch}: Learning rate reduced to {param_group['lr']}")

Lf: 1.33e-03 | Lb: 5.78e-06 | error: 3.72e-03:  50%|██████▍      | 499/1000 [00:51<00:52,  9.46it/s]


--- Iteration 500: Performing Adaptive Grid Update ---


Lf: 4.37e-03 | Lb: 2.92e-06 | error: 2.07e-03:  63%|████████▏    | 626/1000 [01:05<00:40,  9.13it/s]


--- Iteration 625: Performing Adaptive Grid Update ---


Lf: 2.41e-03 | Lb: 7.28e-07 | error: 8.80e-04:  75%|█████████▊   | 751/1000 [01:18<00:25,  9.68it/s]

...

Epoch 2124: Minimum Rel-L2 error reduced to  0.0127%
Epoch 3000: Learning rate reduced to 0.003429999999999999


Lf: 1.36e-05 | Lb: 2.18e-08 | error: 1.17e-04: 100%|████████████| 1000/1000 [01:40<00:00,  9.95it/s]


Epoch 3994: Minimum Rel-L2 error reduced to  0.0080%
Epoch 4000: Learning rate reduced to 0.002400999999999999


Lf: 9.17e-06 | Lb: 3.58e-09 | error: 5.50e-05: 100%|████████████| 1000/1000 [01:44<00:00,  9.52it/s]

Epoch 4998: Minimum Rel-L2 error reduced to  0.0055%
Epoch 5000: Learning rate reduced to 0.0016806999999999992


In [5]:
steps = 1000
for i in range(5):
    train(start_grid_update_step=500, stop_grid_update_step=3000, grid_update_num=20)
    init_epoch = init_epoch + steps
    r_l2 = [loss[3] for loss in loss_history]
    print(f"Epoch {np.argmin(r_l2)}: Minimum Rel-L2 error reduced to {np.min(r_l2)*100: .4f}%")
    for param_group in optimizer.param_groups:
        param_group['lr'] *= 0.7
        print(f"Epoch {init_epoch}: Learning rate reduced to {param_group['lr']}")

Lf: 7.25e-06 | Lb: 2.26e-09 | error: 4.61e-05: 100%|████████████| 1000/1000 [01:46<00:00,  9.38it/s]


Epoch 5999: Minimum Rel-L2 error reduced to  0.0046%
Epoch 6000: Learning rate reduced to 0.0011764899999999994


Lf: 1.12e-05 | Lb: 1.03e-07 | error: 1.70e-04: 100%|████████████| 1000/1000 [01:46<00:00,  9.39it/s]


Epoch 6921: Minimum Rel-L2 error reduced to  0.0042%
Epoch 7000: Learning rate reduced to 0.0008235429999999996


Lf: 2.36e-04 | Lb: 7.30e-08 | error: 3.50e-04: 100%|████████████| 1000/1000 [01:42<00:00,  9.77it/s]


Epoch 7967: Minimum Rel-L2 error reduced to  0.0038%
Epoch 8000: Learning rate reduced to 0.0005764800999999997


Lf: 9.11e-05 | Lb: 2.64e-08 | error: 2.99e-04: 100%|████████████| 1000/1000 [01:39<00:00, 10.06it/s]


Epoch 8967: Minimum Rel-L2 error reduced to  0.0035%
Epoch 9000: Learning rate reduced to 0.00040353606999999974


Lf: 4.09e-06 | Lb: 6.25e-08 | error: 2.73e-04: 100%|████████████| 1000/1000 [01:41<00:00,  9.84it/s]


Epoch 9905: Minimum Rel-L2 error reduced to  0.0033%
Epoch 10000: Learning rate reduced to 0.0002824752489999998
